In [98]:
# Read data into a variable 

words = open("names.txt", "r").read().splitlines()

words[:3]

['emma', 'olivia', 'ava']

In [99]:
# Get all unique characters in words 
chars = sorted(list(set("".join(words))))
chars 

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [100]:
# string to integer 
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi["."] = 0

stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [101]:
# integer to string 
itos = {i:s for s, i in stoi.items()}

itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [102]:
# Create the dataset 
import torch
import torch.nn.functional as F

xs, ys = [], [] 

for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

# convert xs and ys lists to tensors 
xs = torch.tensor(xs)
ys = torch.tensor(ys)

# Get number of training examples from xs 
num = xs.nelement() 
print("number of training examples: ", num)

# Initialize random weights for the network 
g = torch.Generator().manual_seed(2147483547)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# Encode xs to vectors
xenc = F.one_hot(xs, num_classes=27).float() # encode

number of training examples:  228146


In [103]:
# Regularization logic - Which is same thing as label smoothing 
# Similar to how we added 1 to all the bigram counts in other to make sure none of the bigrams has a 0 probability, 
# The weights here represents the bigram counts: 
#  ---- how the matrix multiplication is done
#  ---- How the resulting matrix from the matrix multiplication done below is the linear combination of the rows of the weights based on the rows of xs 

# In order to make sure we have zero probabilities, we need to initialize the weights to be zero or closer to zero. why? 
# if W is zero, then logits will be zero 
# Then counts will be 1 (because exponent of logits which is zero is 1)
# Then the probability probs will be uniform 

# Therefore, 
# Trying to initialize W to be closer to zero as possible is equivalent to label smoothing 
# This leads us to have a regularized loss 

(W**2).mean()

tensor(0.9595, grad_fn=<MeanBackward0>)

In [104]:
# Gradient descent 
for _ in range(300):

    # Forward pass 
    logits = xenc @ W 
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
    print(loss.item())

    # backward pass 
    W.grad = None 
    loss.backward()

    # Update 
    W.data -= 50 * W.grad

3.8475942611694336
3.454775810241699
3.2143616676330566
3.0626327991485596
2.9568722248077393
2.878740072250366
2.81961989402771
2.774364948272705
2.7391722202301025
2.7111282348632812
2.6881418228149414
2.668854236602783
2.652409791946411
2.6382384300231934
2.6259284019470215
2.615163803100586
2.605694055557251
2.5973153114318848
2.5898594856262207
2.5831875801086426
2.577185869216919
2.571758985519409
2.566828489303589
2.562328815460205
2.5582058429718018
2.5544140338897705
2.550915479660034
2.5476770401000977
2.5446712970733643
2.541874647140503
2.539267063140869
2.536830186843872
2.534548282623291
2.5324079990386963
2.530397653579712
2.528505563735962
2.5267229080200195
2.525041103363037
2.523452043533325
2.521949052810669
2.5205256938934326
2.519176721572876
2.5178961753845215
2.5166797637939453
2.5155231952667236
2.5144221782684326
2.5133728981018066
2.5123724937438965
2.511417865753174
2.5105059146881104
2.509633779525757
2.5087995529174805
2.5080010890960693
2.507235527038574
2

In [114]:
# Sample from the NN 

g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = [] 
    ix = 0 
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W 
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print("".join(out))
        

junide.
janasah.
p.
cfay.
a.


In [113]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  out = []
  ix = 0
  while True:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

junide.
janasah.
p.
cfay.
a.
